# Clustering: DBSCAN
Josh Chelen — CISC 683 Final Project

Methods: DBSCAN on RFM features and (separately) the full cleaned dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

DATA_DIR = Path('../data')

---
# Part 1: DBSCAN on RFM Features

## 1a. Load & Scale RFM Data

In [ ]:
rfm = pd.read_csv(DATA_DIR / 'olist_rfm.csv')
print('RFM shape:', rfm.shape)
print(rfm.head())

rfm_features = rfm[['recency', 'frequency', 'monetary']]
scaler_rfm = StandardScaler()
X_rfm = scaler_rfm.fit_transform(rfm_features)
k_rfm = 2 * X_rfm.shape[1]  # min_samples rule of thumb: 2 * n_features

print(f'\nScaled RFM matrix: {X_rfm.shape}')

## 1b. k-Distance Graph — RFM

Sorts distances to the k-th nearest neighbor for each point.
The elbow (where the curve lifts sharply) is the value to use for `eps`.

In [ ]:
nbrs_rfm = NearestNeighbors(n_neighbors=k_rfm, algorithm='ball_tree').fit(X_rfm)
distances_rfm, _ = nbrs_rfm.kneighbors(X_rfm)
distances_rfm = np.sort(distances_rfm[:, k_rfm - 1])

plt.figure(figsize=(8, 4))
plt.plot(distances_rfm)
plt.xlabel('Points (sorted)')
plt.ylabel(f'{k_rfm}-NN Distance')
plt.title('k-Distance Graph — RFM')
plt.tight_layout()
plt.show()

## 1c. Run DBSCAN — RFM

Set `eps_rfm` based on the elbow in the k-distance plot above.

In [ ]:
eps_rfm = 0.5  # adjust based on k-distance plot

db_rfm = DBSCAN(eps=eps_rfm, min_samples=k_rfm, algorithm='ball_tree')
labels_dbscan_rfm = db_rfm.fit_predict(X_rfm)

n_clusters_rfm = len(set(labels_dbscan_rfm)) - (1 if -1 in labels_dbscan_rfm else 0)
n_noise_rfm = (labels_dbscan_rfm == -1).sum()

print(f'[DBSCAN - RFM] eps={eps_rfm}, min_samples={k_rfm}')
print(f'  Clusters found: {n_clusters_rfm}')
print(f'  Noise points:   {n_noise_rfm} ({n_noise_rfm / len(labels_dbscan_rfm):.1%})')

mask_rfm = labels_dbscan_rfm != -1
if n_clusters_rfm >= 2:
    sil_rfm = silhouette_score(X_rfm[mask_rfm], labels_dbscan_rfm[mask_rfm])
    print(f'  Silhouette Score (excl. noise): {sil_rfm:.4f}')
else:
    print('  Not enough clusters for silhouette score — try adjusting eps')

## 1d. Visualize — RFM

In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_rfm[:, 0], X_rfm[:, 2], c=labels_dbscan_rfm,
                      cmap='tab10', s=5, alpha=0.5)
plt.title('DBSCAN — RFM (Recency vs Monetary, scaled)')
plt.xlabel('Recency (scaled)')
plt.ylabel('Monetary (scaled)')
plt.colorbar(scatter, label='Cluster (-1 = noise)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'dbscan_rfm.png', dpi=150)
plt.show()

## 1e. Cluster Profiling — RFM

Mean RFM values and churn rate per cluster (unscaled), to interpret what each cluster represents.

In [ ]:
rfm_profiled = rfm[['recency', 'frequency', 'monetary', 'churn']].copy()
rfm_profiled['cluster'] = labels_dbscan_rfm

print('=== DBSCAN RFM Cluster Profiles (excl. noise) ===')
print(rfm_profiled[rfm_profiled['cluster'] != -1]
      .groupby('cluster')[['recency', 'frequency', 'monetary', 'churn']]
      .mean().round(2))

# Save for downstream use (churn model can use cluster as a feature)
rfm_profiled.to_csv(DATA_DIR / 'olist_rfm_dbscan.csv', index=False)
print('\nSaved: olist_rfm_dbscan.csv')

---
# Part 2: DBSCAN on Full Dataset

> **Note:** This section is independent of Part 1. Run when ready.

## 2a. Load & Scale Full Dataset

In [ ]:
from sklearn.decomposition import PCA

full = pd.read_csv(DATA_DIR / 'olist_cleaned.csv')
print('Full dataset shape:', full.shape)

drop_cols = ['churn'] if 'churn' in full.columns else []
full_features = full.select_dtypes(include=[np.number]).drop(columns=drop_cols, errors='ignore')
full_features = full_features.dropna()

scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(full_features)
k_full = 10  # min_samples for high-dim data

# 2D PCA projection for visualization
pca_vis = PCA(n_components=2)
X_full_2d = pca_vis.fit_transform(X_full)
print(f'Variance captured by 2-component PCA: {pca_vis.explained_variance_ratio_.sum():.2%}')
print(f'Scaled full matrix: {X_full.shape}')

## 2b. k-Distance Graph — Full Dataset

In [ ]:
nbrs_full = NearestNeighbors(n_neighbors=k_full, algorithm='ball_tree').fit(X_full)
distances_full, _ = nbrs_full.kneighbors(X_full)
distances_full = np.sort(distances_full[:, k_full - 1])

plt.figure(figsize=(8, 4))
plt.plot(distances_full)
plt.xlabel('Points (sorted)')
plt.ylabel(f'{k_full}-NN Distance')
plt.title('k-Distance Graph — Full Dataset')
plt.tight_layout()
plt.show()

## 2c. Run DBSCAN — Full Dataset

In [ ]:
eps_full = 5.5  # adjust based on k-distance plot

db_full = DBSCAN(eps=eps_full, min_samples=k_full, algorithm='ball_tree')
labels_dbscan_full = db_full.fit_predict(X_full)

n_clusters_full = len(set(labels_dbscan_full)) - (1 if -1 in labels_dbscan_full else 0)
n_noise_full = (labels_dbscan_full == -1).sum()

print(f'[DBSCAN - Full] eps={eps_full}, min_samples={k_full}')
print(f'  Clusters found: {n_clusters_full}')
print(f'  Noise points:   {n_noise_full} ({n_noise_full / len(labels_dbscan_full):.1%})')

mask_full = labels_dbscan_full != -1
if n_clusters_full >= 2:
    sil_full = silhouette_score(X_full[mask_full], labels_dbscan_full[mask_full])
    print(f'  Silhouette Score (excl. noise): {sil_full:.4f}')
else:
    print('  Not enough clusters for silhouette score — try adjusting eps')

## 2d. Visualize — Full Dataset (PCA 2D projection)

In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_full_2d[:, 0], X_full_2d[:, 1],
                      c=labels_dbscan_full, cmap='tab10', s=3, alpha=0.4)
plt.title('DBSCAN — Full Dataset (PCA 2D projection)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter, label='Cluster (-1 = noise)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'dbscan_full.png', dpi=150)
plt.show()

## 2e. Save Full Dataset Results

In [ ]:
full_profiled = full_features.copy()
full_profiled['cluster'] = labels_dbscan_full
full_profiled.to_csv(DATA_DIR / 'olist_full_dbscan.csv', index=False)
print('Saved: olist_full_dbscan.csv')